In [10]:
import os
import h5py

# Pointing to the cache directory as defined in your code
cache_dir = '../query_cache/'

print(f"Scanning cache directory: {os.path.abspath(cache_dir)}\n")

if not os.path.exists(cache_dir):
    print(f"❌ Directory not found. Please ensure you are running this from the correct folder.")
else:
    # Grab all hdf5 files
    h5_files = [f for f in os.listdir(cache_dir) if f.endswith('.hdf5')]
    
    if not h5_files:
        print("Directory found, but it contains no .hdf5 files.")
    
    for f in h5_files:
        filepath = os.path.join(cache_dir, f)
        print(f"{'='*50}")
        print(f"File: {f}")
        
        try:
            with h5py.File(filepath, 'r') as hf:
                # Check your custom attribute to see how it was saved
                is_dict = hf.attrs.get('IS_DICT', False)
                
                if is_dict:
                    print("Status: Dictionary format (Column names preserved)")
                    print("Columns:")
                    for key in hf.keys():
                        print(f" -> {key}")
                else:
                    num_arrays = hf.attrs.get('NUM_ARRAYS', 'Unknown')
                    print(f"Status: Array format (Column names were NOT preserved during caching)")
                    print(f"Contains {num_arrays} arrays saved as:")
                    for key in hf.keys():
                        print(f" -> {key}")
                        
        except Exception as e:
            print(f"❌ Failed to read {f}. Error: {e}")
        
        print("") # Blank line for readability

Scanning cache directory: /user/sutirtha/query_cache

File: 8016c83936e7ebd827cac7fdd09cce9a80f03abb65850a3c2279bf5189b46759.hdf5
Status: Array format (Column names were NOT preserved during caching)
Contains 1 arrays saved as:
 -> arr_0

File: 8967d2fa752d88d803cac7c4ab098347fefeeb9478fd97f762c6e5b4b7928251.hdf5
Status: Dictionary format (Column names preserved)
Columns:
 -> alpha
 -> alpha_m
 -> aspcapflag
 -> c_fe
 -> ca_fe
 -> ce_fe
 -> fe_h
 -> fe_h_flag
 -> logg
 -> logg_spec
 -> mg_fe
 -> mn_fe
 -> n_fe
 -> na_fe
 -> ni_fe
 -> o_fe
 -> si_fe
 -> starflag
 -> teff
 -> teff_spec
 -> ti_fe
 -> vhelio_avg

File: 3feecb091b1799498d5e8393cea7180e616fdb0242e6f0781a8d8400496fa45e.hdf5
Status: Dictionary format (Column names preserved)
Columns:
 -> c_fe
 -> ca_fe
 -> fe_h
 -> flag_fe_h
 -> flag_sp
 -> logg
 -> mg_fe
 -> rv_comp_1
 -> teff

File: 719b4fbe579eae1bccfc0ebff60cfe5914eace24716d76319e63bd253e3536fb.hdf5
Status: Array format (Column names were NOT preserved during caching)
Cont

In [ ]:
"""
rv_crossmatch.py
================
Cross-matches radial-velocity catalogues for dwarf spheroidal galaxies
against a master membership catalogue (DWG_members.csv).

Output:  DWG_members_with_RV.csv
  • All rows from the master catalogue, with new columns:
        RV_km_s       – heliocentric radial velocity  (km/s)
        e_RV_km_s     – 1-sigma uncertainty           (km/s)
        Membership    – membership probability (Walker/Bootes only)
        Galaxy        – which dSph the measurement comes from
        RV_Source     – literature reference
  • Rows appended at the bottom for RV-catalogue stars that had
    NO positional match in the master (flag column _rv_only = True).

Input catalogues parsed
-----------------------
draco_1.dat          Kleyna+2002    Draco confirmed members   (159 stars)
draco_2.dat          Kleyna+2002    Draco probable members    ( 27 stars)
dwarf_table2.dat     Walker+2009    Carina                    (collapsed per star)
dwarf_table3.dat     Walker+2009    Fornax
dwarf_table4.dat     Walker+2009    Sculptor
dwarf_table5.dat     Walker+2009    Sextans
leo.dat              Mateo+2008     Leo I                     (387 stars)
bootes.dat           Koposov+2011   Bootes I                  (118 stars, if present)
dwarf_catalogue.csv  multi-galaxy   CSV with RA/DEC/v/verr/Pmem (22339 rows, -999=NaN)

All RA/Dec parsed from sexagesimal and stored as decimal degrees (J2000).
Cross-match performed on the unit sphere (chord-length kd-tree, default 3").
"""

import os
import sys
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ─────────────────────────────────────────────────────────────────────────────
# USER SETTINGS  –  edit paths here if needed
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR         = "/user/sutirtha/BallTree_Xmatch"
MASTER_CSV       = os.path.join(DATA_DIR, "DWG_members.csv")
OUTPUT_CSV       = os.path.join(DATA_DIR, "DWG_members_with_RV.csv")
MATCH_RADIUS_AS  = 1.0        # cross-match radius in arcseconds


# ─────────────────────────────────────────────────────────────────────────────
# COORDINATE HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def hms_deg(h, m, s):
    """Hours-minutes-seconds → decimal degrees."""
    return 15.0 * (float(h) + float(m) / 60.0 + float(s) / 3600.0)


def dms_deg(sign, d, m, s):
    """Degrees-arcmin-arcsec (with sign char '+'/'-') → decimal degrees."""
    v = float(d) + float(m) / 60.0 + float(s) / 3600.0
    return -v if str(sign).strip() == "-" else v


def radec_to_xyz(ra_deg, dec_deg):
    """
    Convert arrays of RA/Dec (degrees) to unit-sphere Cartesian coordinates.
    Used for kd-tree distance matching.
    """
    ra  = np.radians(np.asarray(ra_deg,  dtype=float))
    dec = np.radians(np.asarray(dec_deg, dtype=float))
    return np.column_stack([
        np.cos(dec) * np.cos(ra),
        np.cos(dec) * np.sin(ra),
        np.sin(dec),
    ])


# ─────────────────────────────────────────────────────────────────────────────
# PARSERS  (all verified against byte-by-byte catalogue descriptions)
# ─────────────────────────────────────────────────────────────────────────────

# ── Draco  –  Kleyna et al. 2002  (J/MNRAS/330/792) ─────────────────────────
# Fixed-width, 0-indexed character positions:
#  [0:3]   seq number
#  [4:11]  AOP identifier
#  [12:17] Vmag
#  [18:20] RAh   [21:23] RAm   [24:29] RAs
#  [30]    DE-sign
#  [31:33] DEd   [34:36] DEm   [37:41] DEs
#  [42:49] HRV (km/s)   [50:54] e_HRV   [55:59] RTD
def parse_draco(filepath, galaxy="Draco"):
    rows = []
    with open(filepath) as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if len(line) < 49:
                continue
            try:
                rah   = line[18:20].strip()
                ram   = line[21:23].strip()
                ras   = line[24:29].strip()
                sign  = line[30].strip()
                ded   = line[31:33].strip()
                dem   = line[34:36].strip()
                des   = line[37:41].strip()
                hrv   = line[42:49].strip()
                e_hrv = line[50:54].strip() if len(line) > 54 else ""

                if not rah or not hrv:
                    continue

                rows.append({
                    "Galaxy":     galaxy,
                    "RA_deg":     hms_deg(rah, ram, ras),
                    "Dec_deg":    dms_deg(sign or "+", ded, dem, des),
                    "RV_km_s":    float(hrv),
                    "e_RV_km_s":  float(e_hrv) if e_hrv else np.nan,
                    "Membership": np.nan,
                    "Source":     "Kleyna+2002",
                })
            except Exception:
                continue
    return pd.DataFrame(rows)


# ── Walker+2009  –  Carina / Fornax / Sculptor / Sextans ─────────────────────
# Fixed-width, 0-indexed character positions:
#  [0:8]   Target ID (e.g. "Car-0001")
#  [9:13]  Field+channel
#  [14:22] HJD
#  [23:25] RAh   [26:28] RAm   [29:34] RAs
#  [35]    DE-sign
#  [36:38] DEd   [39:41] DEm   [42:46] DEs
#  [47:52] Vmag  [53:58] V-I
#  [59:65] HV (single obs.)   [66:70] e_HV
#  [71:76] SigFe  [77:81] e_SigFe  [82:87] SigMg  [88:92] e_SigMg
#  [93:98] Mmb  (membership probability)
#  [99:104] <HV>  (weighted mean)   [105:108] e_<HV>
#  [109:114] <SigMg>               [115:119] e_<SigMg>
#
# Multiple observations exist per star → collapse to one row using the
# catalogue's own weighted-mean column (<HV>) where present; otherwise
# compute inverse-variance weighted mean from individual epochs.

WALKER_GALAXY_MAP = {
    "dwarf_table2.dat": "Carina",
    "dwarf_table3.dat": "Fornax",
    "dwarf_table4.dat": "Sculptor",
    "dwarf_table5.dat": "Sextans",
}

def parse_walker(filepath):
    galaxy = WALKER_GALAXY_MAP.get(os.path.basename(filepath), "Unknown")
    rows = []
    with open(filepath) as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if len(line) < 70:
                continue
            try:
                target   = line[0:8].strip()
                rah      = line[23:25].strip()
                ram      = line[26:28].strip()
                ras      = line[29:34].strip()
                sign     = line[35] if len(line) > 35 else "+"
                ded      = line[36:38].strip()
                dem      = line[39:41].strip()
                des      = line[42:46].strip()
                hv       = line[59:65].strip()
                e_hv     = line[66:70].strip()
                mmb      = line[93:98].strip()   if len(line) > 98  else ""
                mean_hv  = line[99:104].strip()  if len(line) > 104 else ""
                e_mean   = line[105:108].strip() if len(line) > 108 else ""

                if not rah or not hv:
                    continue

                rows.append({
                    "Target":       target,
                    "Galaxy":       galaxy,
                    "RA_deg":       hms_deg(rah, ram, ras),
                    "Dec_deg":      dms_deg(sign.strip() or "+", ded, dem, des),
                    "HV_single":    float(hv),
                    "e_HV_single":  float(e_hv)    if e_hv   else np.nan,
                    "Membership":   float(mmb)     if mmb    else np.nan,
                    "Mean_HV":      float(mean_hv) if mean_hv else np.nan,
                    "e_Mean_HV":    float(e_mean)  if e_mean  else np.nan,
                    "Source":       "Walker+2009",
                })
            except Exception:
                continue

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Collapse multiple epochs → one representative velocity per star
    def best_rv(grp):
        # Prefer catalogue's own weighted-mean column
        has_mean = grp.dropna(subset=["Mean_HV"])
        if not has_mean.empty:
            best = has_mean.iloc[0]
            rv, erv = best["Mean_HV"], best["e_Mean_HV"]
        else:
            # Compute inverse-variance weighted mean from single-epoch rows
            sub = grp.dropna(subset=["HV_single", "e_HV_single"])
            if not sub.empty:
                w   = 1.0 / sub["e_HV_single"] ** 2
                rv  = (sub["HV_single"] * w).sum() / w.sum()
                erv = 1.0 / np.sqrt(w.sum())
            else:
                rv, erv = grp["HV_single"].mean(), np.nan

        mmb = grp["Membership"].dropna()
        return pd.Series({
            "Galaxy":     grp["Galaxy"].iloc[0],
            "RA_deg":     grp["RA_deg"].iloc[0],
            "Dec_deg":    grp["Dec_deg"].iloc[0],
            "RV_km_s":    rv,
            "e_RV_km_s":  erv,
            "Membership": mmb.iloc[0] if not mmb.empty else np.nan,
            "Source":     "Walker+2009",
        })

    return df.groupby("Target", sort=False).apply(best_rv).reset_index(drop=True)


# ── Leo I  –  Mateo et al. 2008  (J/ApJ/675/201) ────────────────────────────
# Fixed-width, 0-indexed character positions:
#  [0:3]   Seq   [3] flag
#  [5:11]  R (arcsec from center)   [12:17] PA
#  [18:24] HV (km/s)   [25:28] sigma
#  [29:31] RAh   [32:34] RAm   [35:40] RAs
#  [41]    DE-sign
#  [42:44] DEd   [45:47] DEm   [48:52] DEs
#  [54:59] Imag  [60:64] V-I   [65:76] r_HV source codes
def parse_leo(filepath, galaxy="LeoI"):
    rows = []
    with open(filepath) as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if len(line) < 52:
                continue
            try:
                hv   = line[18:24].strip()
                e_hv = line[25:28].strip()
                rah  = line[29:31].strip()
                ram  = line[32:34].strip()
                ras  = line[35:40].strip()
                sign = line[41] if len(line) > 41 else "+"
                ded  = line[42:44].strip()
                dem  = line[45:47].strip()
                des  = line[48:52].strip()

                if not rah or not hv:
                    continue

                rows.append({
                    "Galaxy":     galaxy,
                    "RA_deg":     hms_deg(rah, ram, ras),
                    "Dec_deg":    dms_deg(sign.strip() or "+", ded, dem, des),
                    "RV_km_s":    float(hv),
                    "e_RV_km_s":  float(e_hv) if e_hv else np.nan,
                    "Membership": np.nan,
                    "Source":     "Mateo+2008",
                })
            except Exception:
                continue
    return pd.DataFrame(rows)


# ── Bootes I  –  Koposov et al. 2011  (J/ApJ/736/146) ───────────────────────
# Fixed-width, 0-indexed character positions:
#  [0:3]   Seq
#  [4:23]  SDSS name  (JHHMMSS.ss+DDMMSS.s)
#  [24:32] RAdeg  (decimal degrees – already converted!)
#  [33:40] DEdeg  (decimal degrees)
#  [41:45] gmag   [46:50] rmag
#  [51:58] RV (km/s)    [59:63] e_RV
#  [64:68] Pvar   [69:73] Teff   [74:78] [Fe/H]   [79:82] logg
#  [83]    Mm flag  ('B' = best member)
def parse_bootes(filepath, galaxy="BooI"):
    rows = []
    with open(filepath) as fh:
        for raw in fh:
            line = raw.rstrip("\n")
            if len(line) < 63:
                continue
            try:
                ra_s  = line[24:32].strip()
                dec_s = line[33:40].strip()
                rv    = line[51:58].strip()
                e_rv  = line[59:63].strip()
                mm    = line[83].strip() if len(line) >= 84 else ""

                if not ra_s or not rv or rv == "-":
                    continue

                rows.append({
                    "Galaxy":     galaxy,
                    "RA_deg":     float(ra_s),      # already decimal degrees
                    "Dec_deg":    float(dec_s),
                    "RV_km_s":    float(rv),
                    "e_RV_km_s":  float(e_rv) if (e_rv and e_rv != "-") else np.nan,
                    "Membership": 1.0 if mm == "B" else np.nan,
                    "Source":     "Koposov+2011",
                })
            except Exception:
                continue
    return pd.DataFrame(rows)


# ── dwarf_catalogue.csv  –  multi-galaxy CSV catalogue ──────────────────────
# Columns (all already in decimal degrees, no conversion needed):
#   Galaxy   – galaxy name string  (e.g. "Aqr2", "Draco", "Carina" …)
#   RA       – Right Ascension, decimal degrees (J2000)
#   DEC      – Declination,     decimal degrees (J2000)
#   v        – heliocentric radial velocity  (km/s);  -999 = missing
#   verr     – velocity uncertainty          (km/s);  -999 = missing
#   Pmem     – membership probability  [0,1]; -999 = missing
#   (r, gr, nmask, t_exp, SN, CaT, CaTerr, FeH, FeH_err, Var also present)
def parse_dwarf_catalogue(filepath):
    df = pd.read_csv(filepath, low_memory=False)

    # Replace -999 sentinel with NaN across all numeric columns
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace(-999, np.nan)
    df[num_cols] = df[num_cols].replace(-999.0, np.nan)

    # Must have RA, DEC, v at minimum
    required = {"RA", "DEC", "v"}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f"dwarf_catalogue.csv is missing expected columns: {missing}\n"
            f"Found: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "Galaxy":     df["Galaxy"].astype(str).str.strip(),
        "RA_deg":     pd.to_numeric(df["RA"],   errors="coerce"),
        "Dec_deg":    pd.to_numeric(df["DEC"],  errors="coerce"),
        "RV_km_s":    pd.to_numeric(df["v"],    errors="coerce"),
        "e_RV_km_s":  pd.to_numeric(df["verr"], errors="coerce") if "verr" in df.columns else np.nan,
        "Membership": pd.to_numeric(df["Pmem"], errors="coerce") if "Pmem" in df.columns else np.nan,
        "Source":     "dwarf_catalogue",
    })

    # Drop rows with no usable RV or no position
    out = out.dropna(subset=["RA_deg", "Dec_deg", "RV_km_s"]).reset_index(drop=True)
    return out


# ─────────────────────────────────────────────────────────────────────────────
# MASTER CATALOGUE  –  auto-detect RA/Dec columns, convert to decimal degrees
# ─────────────────────────────────────────────────────────────────────────────
_RA_CANDIDATES  = ["ra_x", "ra", "ra_deg", "_raj2000", "raj2000", "ra_j2000",
                   "ra(deg)", "alpha", "ra_icrs"]
_DEC_CANDIDATES = ["dec_x", "dec", "de", "dec_deg", "_dej2000", "dej2000",
                   "dec_j2000", "dec(deg)", "delta", "de_icrs"]

def _find_col(columns, candidates):
    """Case-insensitive column search."""
    lc = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lc:
            return lc[cand]
    return None


def load_master(csv_path):
    """
    Load master CSV, find RA/Dec columns, convert to decimal degrees.
    Adds columns  RA_master  and  Dec_master  (decimal degrees, J2000).
    """
    df = pd.read_csv(csv_path, low_memory=False)
    print(f"  Rows : {len(df)}")
    print(f"  Cols : {list(df.columns)}")

    ra_col  = _find_col(df.columns, _RA_CANDIDATES)
    dec_col = _find_col(df.columns, _DEC_CANDIDATES)

    if ra_col is None or dec_col is None:
        # Last resort: print columns and raise so the user can fix
        print("\n  [ERROR] Could not identify RA/Dec columns automatically.")
        print(f"  Please rename your RA column to 'RA' and Dec column to 'Dec'.")
        sys.exit(1)

    print(f"  RA  column → '{ra_col}'")
    print(f"  Dec column → '{dec_col}'")

    # Detect whether values are already decimal floats or sexagesimal strings
    sample_ra = str(df[ra_col].dropna().iloc[0])
    try:
        float(sample_ra)                       # numeric → assume decimal degrees
        df["RA_master"]  = pd.to_numeric(df[ra_col],  errors="coerce")
        df["Dec_master"] = pd.to_numeric(df[dec_col], errors="coerce")
        print("  Format : decimal degrees (no conversion needed)")
    except ValueError:
        # Sexagesimal string – parse with regex
        import re
        def _hms(s):
            p = re.split(r"[hm:\s]+", str(s).replace("s","").strip())
            p = [x for x in p if x]
            return hms_deg(*p[:3]) if len(p) >= 3 else np.nan

        def _dms(s):
            s = str(s).strip()
            sign = "-" if s.startswith("-") else "+"
            p = re.split(r"[dm:\s]+", s.lstrip("+-").replace("s","").strip())
            p = [x for x in p if x]
            return dms_deg(sign, *p[:3]) if len(p) >= 3 else np.nan

        df["RA_master"]  = df[ra_col].apply(_hms)
        df["Dec_master"] = df[dec_col].apply(_dms)
        print("  Format : sexagesimal strings (converted to decimal degrees)")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# CROSS-MATCH  (unit-sphere kd-tree, no astropy required)
# ─────────────────────────────────────────────────────────────────────────────
def crossmatch(master_df, rv_df, radius_arcsec=MATCH_RADIUS_AS):
    """
    Positional cross-match of rv_df against master_df.

    Algorithm
    ---------
    1. Convert both catalogues to 3-D unit-sphere Cartesian (cos/sin).
    2. Build cKDTree on master; query once for all RV sources.
    3. Accept matches with chord distance ≤ 2·sin(θ/2) for the given θ.

    Returns
    -------
    master_out  : master_df with RV columns filled where matched
    unmatched   : rows of rv_df that found no counterpart in master
    """
    master_ra  = master_df["RA_master"].values
    master_dec = master_df["Dec_master"].values
    rv_ra      = rv_df["RA_deg"].values
    rv_dec     = rv_df["Dec_deg"].values

    # Drop master rows with NaN coords from tree (keep track of which are valid)
    valid_master = np.isfinite(master_ra) & np.isfinite(master_dec)
    xyz_master   = radec_to_xyz(master_ra[valid_master], master_dec[valid_master])
    master_valid_idx = np.where(valid_master)[0]   # maps tree index → df index

    valid_rv   = np.isfinite(rv_ra) & np.isfinite(rv_dec)
    xyz_rv     = radec_to_xyz(rv_ra[valid_rv], rv_dec[valid_rv])
    rv_valid_idx = np.where(valid_rv)[0]

    # Chord-length threshold for the requested angular separation
    theta_rad  = np.radians(radius_arcsec / 3600.0)
    chord_max  = 2.0 * np.sin(theta_rad / 2.0)

    tree       = cKDTree(xyz_master)
    dist, tidx = tree.query(xyz_rv, k=1, distance_upper_bound=chord_max * 1.5)
    hit        = dist <= chord_max                 # boolean mask over valid RV rows

    n_match    = hit.sum()
    n_nomatch  = (~hit).sum()
    print(f"  Total RV stars   : {len(rv_df)}")
    print(f"  Matched          : {n_match}")
    print(f"  Unmatched (new)  : {n_nomatch}")

    # ── Initialise output columns in master ──────────────────────────────────
    out = master_df.copy()
    for col, fill in [("RV_km_s", np.nan), ("e_RV_km_s", np.nan),
                      ("Membership", np.nan),
                      ("Galaxy", ""), ("RV_Source", ""),
                      ("_rv_only", False)]:
        if col not in out.columns:
            out[col] = fill

    # ── Write matched RV values into master rows ──────────────────────────────
    for i, is_hit in enumerate(hit):
        if not is_hit:
            continue
        rv_row_idx     = rv_valid_idx[i]        # original index in rv_df
        master_row_idx = master_valid_idx[tidx[i]]   # original index in master_df

        rv_row = rv_df.iloc[rv_row_idx]

        # Only fill if still empty (first match wins; avoids overwriting)
        if pd.isna(out.at[master_row_idx, "RV_km_s"]):
            out.at[master_row_idx, "RV_km_s"]    = rv_row["RV_km_s"]
            out.at[master_row_idx, "e_RV_km_s"]  = rv_row.get("e_RV_km_s", np.nan)
            out.at[master_row_idx, "Membership"]  = rv_row.get("Membership", np.nan)
            out.at[master_row_idx, "Galaxy"]      = rv_row.get("Galaxy", "")
            out.at[master_row_idx, "RV_Source"]   = rv_row.get("Source", "")

    # ── Collect unmatched RV stars ────────────────────────────────────────────
    unmatched_rv_idx = rv_valid_idx[~hit]                         # df indices
    unmatched = rv_df.iloc[unmatched_rv_idx].copy().reset_index(drop=True)
    unmatched = unmatched.rename(columns={
        "RA_deg":    "RA_master",
        "Dec_deg":   "Dec_master",
        "Membership": "Membership",
        "Source":    "RV_Source",
    })
    # Galaxy column already named correctly; add flag
    unmatched["_rv_only"] = True

    # Keep only rows where RA, Dec, RV *and* e_RV are all non-null
    before = len(unmatched)
    unmatched = unmatched.dropna(
        subset=["RA_master", "Dec_master", "RV_km_s", "e_RV_km_s"]
    ).reset_index(drop=True)
    after = len(unmatched)
    if before - after > 0:
        print(f"  Dropped {before - after} unmatched rows missing RA/Dec/RV/e_RV "
              f"→ {after} appended")

    return out, unmatched


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    os.makedirs(DATA_DIR, exist_ok=True)

    # ── 1. Parse RV catalogues ────────────────────────────────────────────────
    print("\n" + "="*60)
    print("STEP 1 – Parsing RV catalogues")
    print("="*60)

    catalogue_specs = [
        ("Draco (members)",    parse_draco,           "draco_1.dat",          {"galaxy": "Draco"}),
        ("Draco (probable)",   parse_draco,           "draco_2.dat",          {"galaxy": "Draco"}),
        ("Carina",             parse_walker,          "dwarf_table2.dat",     {}),
        ("Fornax",             parse_walker,          "dwarf_table3.dat",     {}),
        ("Sculptor",           parse_walker,          "dwarf_table4.dat",     {}),
        ("Sextans",            parse_walker,          "dwarf_table5.dat",     {}),
        ("Leo I",              parse_leo,             "leo.dat",              {"galaxy": "LeoI"}),
        ("Bootes I",           parse_bootes,          "bootes.dat",           {"galaxy": "BooI"}),
        ("dwarf_catalogue",    parse_dwarf_catalogue, "dwarf_catalogue.csv",  {}),
    ]

    frames = []
    for label, parser, filename, kwargs in catalogue_specs:
        path = os.path.join(DATA_DIR, filename)
        if not os.path.exists(path):
            print(f"  {label:<22s}  ⚠  FILE NOT FOUND – skipped ({filename})")
            continue
        df = parser(path, **kwargs)
        print(f"  {label:<22s}  {len(df):5d} stars  →  {filename}")
        if not df.empty:
            frames.append(df)

    if not frames:
        print("\n[FATAL] No RV data parsed. Check DATA_DIR path.")
        sys.exit(1)

    # Standardise to a common column set
    KEEP = ["Galaxy", "RA_deg", "Dec_deg", "RV_km_s", "e_RV_km_s",
            "Membership", "Source"]
    all_rv = pd.concat(
        [f.reindex(columns=KEEP) for f in frames],
        ignore_index=True,
    )

    print(f"\n  Total unique RV stars (all galaxies): {len(all_rv)}")
    print("\n  Per-galaxy summary:")
    smry = all_rv.groupby("Galaxy").agg(
        N_stars=("RV_km_s", "count"),
        Mean_RV=("RV_km_s", "mean"),
        Std_RV =("RV_km_s", "std"),
    ).round(2)
    print(smry.to_string())

    # ── 2. Load master catalogue ───────────────────────────────────────────────
    print("\n" + "="*60)
    print("STEP 2 – Loading master catalogue")
    print("="*60)

    if not os.path.exists(MASTER_CSV):
        print(f"\n  ⚠  Master catalogue NOT FOUND at:\n     {MASTER_CSV}")
        print("\n  → Saving combined RV table as standalone output instead.")
        all_rv.to_csv(OUTPUT_CSV, index=False)
        print(f"\n  Output → {OUTPUT_CSV}  ({len(all_rv)} rows)")
        return

    master = load_master(MASTER_CSV)

    # ── 3. Cross-match ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"STEP 3 – Cross-matching  (radius = {MATCH_RADIUS_AS}\")")
    print("="*60)
    master_out, unmatched = crossmatch(master, all_rv)

    # ── 4. Build final catalogue ───────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("STEP 4 – Building final output")
    print("="*60)

    final = pd.concat([master_out, unmatched], ignore_index=True, sort=False)

    rv_in_master  = master_out["RV_km_s"].notna().sum()
    rv_appended   = len(unmatched)
    print(f"  Master rows total          : {len(master_out)}")
    print(f"    • with RV matched        : {rv_in_master}")
    print(f"    • without RV             : {master_out['RV_km_s'].isna().sum()}")
    print(f"  Appended RV-only rows      : {rv_appended}  (_rv_only = True)")
    print(f"  ──────────────────────────────")
    print(f"  TOTAL output rows          : {len(final)}")

    # ── 5. Save ────────────────────────────────────────────────────────────────
    final.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓  Saved → {OUTPUT_CSV}")

    # Per-galaxy fill rate
    if "Galaxy" in final.columns and final["Galaxy"].notna().any():
        print("\n  RV fill rate per galaxy (master rows only):")
        master_rows = final[~final["_rv_only"]]
        for gal, grp in master_rows[master_rows["Galaxy"] != ""].groupby("Galaxy"):
            n_tot  = len(grp)
            n_rv   = grp["RV_km_s"].notna().sum()
            print(f"    {gal:<12s}: {n_rv:4d} / {n_tot:4d} matched")


if __name__ == "__main__":
    main()


STEP 1 – Parsing RV catalogues
  Draco (members)           159 stars  →  draco_1.dat
  Draco (probable)           27 stars  →  draco_2.dat


/tmp/ipykernel_2342997/946321118.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("Target", sort=False).apply(best_rv).reset_index(drop=True)


  Carina                   1982 stars  →  dwarf_table2.dat


/tmp/ipykernel_2342997/946321118.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("Target", sort=False).apply(best_rv).reset_index(drop=True)


  Fornax                   2633 stars  →  dwarf_table3.dat


/tmp/ipykernel_2342997/946321118.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("Target", sort=False).apply(best_rv).reset_index(drop=True)


  Sculptor                 1541 stars  →  dwarf_table4.dat


/tmp/ipykernel_2342997/946321118.py:223: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("Target", sort=False).apply(best_rv).reset_index(drop=True)


  Sextans                   947 stars  →  dwarf_table5.dat
  Leo I                     387 stars  →  leo.dat
  Bootes I                  112 stars  →  bootes.dat
  dwarf_catalogue         22339 stars  →  dwarf_catalogue.csv

  Total unique RV stars (all galaxies): 30127

  Per-galaxy summary:
          N_stars  Mean_RV  Std_RV
Galaxy                            
Aqr2           16   -88.94   69.07
Aqr3           19   -54.35   75.29
Boo1          236    49.52  108.09
Boo2          181   -17.28  120.40
Boo3          247    -5.55  133.63
BooI          112    67.26   76.11
CB            199    55.05   91.56
CVn1          367    17.28   47.97
CVn2          113   -61.58   80.45
Carina       1982   136.62   90.01
Cet3           30   -35.24   70.14
Col1           72    77.25   53.35
Dra          1644  -254.37   96.68
Dra2           97  -184.35  141.32
Draco         186  -291.74   10.73
Eri           118    32.16   73.35
Eri4           36    -8.86   77.65
For           737    52.43   26.95
Fornax

In [3]:
import pandas as pd
df = pd.read_csv('/user/sutirtha/BallTree_Xmatch/dwarf_catalogue.csv', low_memory=False)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print()
print(df.head(3).to_string())

Shape: (22339, 16)
Columns: ['Galaxy', 'RA', 'DEC', 'r', 'gr', 'nmask', 't_exp', 'SN', 'v', 'verr', 'CaT', 'CaTerr', 'FeH', 'FeH_err', 'Var', 'Pmem']

  Galaxy          RA       DEC          r        gr  nmask   t_exp         SN          v      verr       CaT    CaTerr         FeH     FeH_err    Var      Pmem
0   Aqr2  338.478833 -9.287944  18.069878  0.807788      1  3600.0  56.680179  49.479820  1.125074  6.631335  0.171823 -999.000000 -999.000000 -999.0  0.000000
1   Aqr2  338.469542 -9.285833  18.537954  0.717903      1  3600.0  44.841800 -56.558532  1.217907  2.851551  0.149202   -2.558493    0.142758 -999.0  0.849491
2   Aqr2  338.460833 -9.315222  19.513830  0.601021      1  3600.0  23.918729 -68.371054  1.553705  2.281667  0.236711   -2.690356    0.171853 -999.0  0.752323
